SLR SGD From Scratch

In [3]:
import numpy as np
import pandas as pd


class SLR_SGD:

    # =====================================================
    # CONSTRUCTOR
    # =====================================================
    def __init__(self, learning_rate=0.001, epochs=1000):

        self.learning_rate = learning_rate
        self.epochs = epochs

        # Parameters
        self.coef_ = 0
        self.intercept_ = 0

    # =====================================================
    # TRAIN TEST SPLIT
    # =====================================================
    def train_test_split(self, X, y, test_size=0.2, random_state=None):

        X = np.array(X)
        y = np.array(y)

        n = len(X)

        indices = np.arange(n)

        np.random.seed(random_state)
        np.random.shuffle(indices)

        X = X[indices]
        y = y[indices]

        split_index = int(n * (1 - test_size))

        X_train = X[:split_index]
        X_test = X[split_index:]

        y_train = y[:split_index]
        y_test = y[split_index:]

        return X_train, X_test, y_train, y_test

    # =====================================================
    # FIT MODEL (Stochastic Gradient Descent)
    # =====================================================
    def fit(self, X_train, y_train):

        X_train = np.array(X_train).reshape(-1)
        y_train = np.array(y_train)

        n = len(X_train)

        # Epoch loop
        for epoch in range(self.epochs):

            # Loop through every row
            for i in range(n):

                xi = X_train[i]
                yi = y_train[i]

                # Prediction
                y_pred = (
                    self.intercept_
                    + (self.coef_ * xi)
                )

                # Error
                error = yi - y_pred

                # Gradients
                d_intercept = -2 * error

                d_coef = -2 * xi * error

                # Update Parameters
                self.intercept_ = (
                    self.intercept_
                    - self.learning_rate * d_intercept
                )

                self.coef_ = (
                    self.coef_
                    - self.learning_rate * d_coef
                )

    # =====================================================
    # PREDICT
    # =====================================================
    def predict(self, X_test):

        X_test = np.array(X_test).reshape(-1)

        y_pred = (
            self.intercept_
            + (self.coef_ * X_test)
        )

        return y_pred

    # =====================================================
    # MSE
    # =====================================================
    def mean_squared_error(self, y_true, y_pred):

        y_true = np.array(y_true)
        y_pred = np.array(y_pred)

        mse = np.mean((y_true - y_pred) ** 2)

        return mse

    # =====================================================
    # MAE
    # =====================================================
    def mean_absolute_error(self, y_true, y_pred):

        y_true = np.array(y_true)
        y_pred = np.array(y_pred)

        mae = np.mean(np.abs(y_true - y_pred))

        return mae

    # =====================================================
    # R2 SCORE
    # =====================================================
    def r2_score(self, y_true, y_pred):

        y_true = np.array(y_true)
        y_pred = np.array(y_pred)

        ss_residual = np.sum(
            (y_true - y_pred) ** 2
        )

        ss_total = np.sum(
            (y_true - np.mean(y_true)) ** 2
        )

        r2 = 1 - (ss_residual / ss_total)

        return r2

    # =====================================================
    # ADJUSTED R2
    # =====================================================
    def adjusted_r2_score(self, y_true, y_pred, X_test):

        r2 = self.r2_score(y_true, y_pred)

        n = X_test.shape[0]

        if len(X_test.shape) == 1:
            p = 1
        else:
            p = X_test.shape[1]

        adjusted_r2 = 1 - (
            ((1 - r2) * (n - 1)) / (n - p - 1)
        )

        return adjusted_r2

    # =====================================================
    # COMPLETE EVALUATION
    # =====================================================
    def evaluation(self, y_true, y_pred, X_test):

        mse = self.mean_squared_error(
            y_true,
            y_pred
        )

        rmse = np.sqrt(mse)

        mae = self.mean_absolute_error(
            y_true,
            y_pred
        )

        r2 = self.r2_score(
            y_true,
            y_pred
        )

        adjusted_r2 = self.adjusted_r2_score(
            y_true,
            y_pred,
            X_test
        )

        print("MSE          :", mse)
        print("RMSE         :", rmse)
        print("MAE          :", mae)
        print("R2 Score     :", r2)
        print("Adjusted R2  :", adjusted_r2)




In [5]:
# =====================================================
# EXAMPLE USAGE
# =====================================================

# Dataset
data = pd.read_csv("placement.csv")

# Features and Target
X = data[['CGPA']]
y = data['Package_LPA']

# Model
model = SLR_SGD(
    learning_rate=0.001,
    epochs=1000
)

# Split
X_train, X_test, y_train, y_test = model.train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Train
model.fit(X_train, y_train)

# Prediction
y_pred = model.predict(X_test)

# Parameters
print("Intercept :", model.intercept_)
print("Slope     :", model.coef_)

# Evaluation
model.evaluation(y_test, y_pred, X_test)

Intercept : -1.8519813358089023
Slope     : 1.0128081524178667
MSE          : 0.01332357289555808
RMSE         : 0.11542778216511863
MAE          : 0.10464766283226576
R2 Score     : 0.9887476499651885
Adjusted R2  : 0.9881850324634479


In [ ]:
# Why scratch SGD differs from sklearn SGDRegressor

# 1. sklearn uses L2 regularization by default
# 2. sklearn uses different learning rate strategy
# 3. sklearn shuffles data every epoch
# 4. sklearn uses convergence checks
# 5. sklearn expects scaled data

# My scratch SGD uses simple constant learning rate
# without regularization, so results differ.